# 🏠 House Price Prediction
### Internship Project — Week 1
**Assigned:** 17 May 2026 &nbsp;&nbsp;|&nbsp;&nbsp; **Submitted:** 21 May 2026  
**Dataset:** [Housing Prices Dataset — Kaggle (yasserh)](https://www.kaggle.com/datasets/yasserh/housing-prices-dataset)

---

## 🎯 Problem Statement

Real estate buyers and sellers often rely on guesswork or outdated comparisons to estimate a property's fair value.  
This project builds a **regression model** that predicts house prices based on features such as area, rooms, amenities and location — and identifies which features most strongly influence price.

**Tools used:** Python 3 · Pandas · Scikit-learn · Matplotlib · Seaborn

---

## 📂 Task 1 — Data Loading & Exploration

### 1.1 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 120
print("Libraries imported successfully")

### 1.2 Load Dataset & Display First 10 Rows

In [ ]:
df = pd.read_csv('/kaggle/input/housing-prices-dataset/Housing.csv')
print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head(10)

### 1.3 Rows & Columns

In [ ]:
print(f"Total Rows   : {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")

### 1.4 Target vs Feature Columns

In [ ]:
print("🎯 Target  : price")
print("📋 Features:")
for col in df.columns:
    if col != 'price':
        print(f"   • {col:<20}  dtype: {df[col].dtype}")

🎯 Target  : price
📋 Features:
   • area                  dtype: int64
   • bedrooms              dtype: int64
   • bathrooms             dtype: int64
   • stories               dtype: int64
   • mainroad              dtype: object
   • guestroom             dtype: object
   • basement              dtype: object
   • hotwaterheating       dtype: object
   • airconditioning       dtype: object
   • parking               dtype: int64
   • prefarea              dtype: object
   • furnishingstatus      dtype: object


### 1.5 Check for Missing Values

In [ ]:
print("=== Missing Values per Column ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "No missing values found.")
print(f"\nTotal missing cells: {df.isnull().sum().sum()}")

=== Missing Values per Column ===
No missing values found.

Total missing cells: 0


### 1.6 Statistical Summary

In [ ]:
df.describe()

,price,area,bedrooms,bathrooms,stories,parking
count,5.450000e+02,545.000000,545.000000,545.000000,545.000000,545.000000
mean,4.766729e+06,5150.541284,2.965138,1.286239,1.805505,0.693578
std,1.870440e+06,2170.141023,0.738064,0.502470,0.867492,0.861586
min,1.750000e+06,1650.000000,1.000000,1.000000,1.000000,0.000000
25%,3.430000e+06,3600.000000,2.000000,1.000000,1.000000,0.000000
50%,4.340000e+06,4600.000000,3.000000,1.000000,2.000000,0.000000
75%,5.740000e+06,6360.000000,3.000000,2.000000,2.000000,1.000000
max,1.330000e+07,16200.000000,6.000000,4.000000,4.000000,3.000000


---
## 🧹 Task 2 — Data Cleaning

### 2.1 Handle Missing Values

In [ ]:
# Fill numeric columns with median (safety step — dataset has no NaN but good practice)
for col in df.select_dtypes(include='number').columns:
    n_miss = df[col].isnull().sum()
    if n_miss > 0:
        med = df[col].median()
        df[col] = df[col].fillna(med)
        print(f"Filled '{col}': {n_miss} NaN → median ({med:.1f})")

assert df.isnull().sum().sum() == 0, "NaN values still present!"
print(f"✅ Zero NaN values remaining.")

### 2.2 Remove Duplicate Rows

In [ ]:
before = len(df)
df = df.drop_duplicates()
print(f"Duplicates removed: {before - len(df)}  ({before} → {len(df)} rows)")

Duplicates removed: 0  (545 → 545 rows)


### 2.3 Encode Categorical Columns

Binary `yes/no` columns are mapped to `0/1`.  
`furnishingstatus` is **ordinally encoded** (unfurnished=0, semi-furnished=1, furnished=2) since it has a natural order.

In [ ]:
# Encode binary yes/no columns → 0/1
binary_cols = ['mainroad','guestroom','basement','hotwaterheating','airconditioning','prefarea']
for col in binary_cols:
    df[col] = df[col].map({'yes': 1, 'no': 0})
    print(f"Encoded '{col}'  →  0/1")

# Ordinal encode furnishingstatus
df['furnishingstatus'] = df['furnishingstatus'].map(
    {'furnished': 2, 'semi-furnished': 1, 'unfurnished': 0}
)
print("Encoded 'furnishingstatus'  →  0/1/2")

Encoded 'mainroad'  →  0/1
Encoded 'guestroom'  →  0/1
Encoded 'basement'  →  0/1
Encoded 'hotwaterheating'  →  0/1
Encoded 'airconditioning'  →  0/1
Encoded 'prefarea'  →  0/1
Encoded 'furnishingstatus'  →  0/1/2


### 2.4 Verify Cleaned Dataset

In [ ]:
print("=== Cleaned Dataset — Data Types ===")
print(df.dtypes)
print(f"\nAny NaN remaining: {df.isnull().sum().sum()}")
df.head(5)

=== Cleaned Dataset — Data Types ===
price               int64
area                int64
bedrooms            int64
bathrooms           int64
stories             int64
mainroad            int64
guestroom           int64
basement            int64
hotwaterheating     int64
airconditioning     int64
parking             int64
prefarea            int64
furnishingstatus    int64
dtype: object

Any NaN remaining: 0


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,1,0,0,0,1,2,1,2
1,12250000,8960,4,4,4,1,0,0,0,1,3,0,2
2,12250000,9960,3,2,2,1,0,1,0,0,2,1,1
3,12215000,7500,4,2,2,1,0,1,0,1,3,1,2
4,11410000,7420,4,1,2,1,1,1,0,1,2,0,2


---
## 🤖 Task 3 — Model Building & Evaluation

### 3.1 Define Features and Target

In [ ]:
X = df.drop('price', axis=1)
y = df['price']
print(f"Feature matrix : {X.shape}")
print(f"Target vector  : {y.shape}")
print(f"\nFeatures: {list(X.columns)}")

### 3.2 Train / Test Split (80 / 20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Train : {len(X_train)} samples")
print(f"Test  : {len(X_test)} samples")

Train : 436 samples
Test  : 109 samples


### 3.3 Evaluation Helper Function

In [ ]:
def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"\n{'='*42}")
    print(f"  Model : {name}")
    print(f"  MAE   : {mae:>12,.0f}")
    print(f"  RMSE  : {rmse:>12,.0f}")
    print(f"  R²    : {r2:>12.4f}  ({r2*100:.1f}% variance explained)")
    print(f"{'='*42}")
    return mae, rmse, r2

### 3.4 Model 1 — Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
mae_lr, rmse_lr, r2_lr = evaluate("Linear Regression", y_test, y_pred_lr)

### 3.5 Model 2 — Random Forest Regressor

In [ ]:
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
mae_rf, rmse_rf, r2_rf = evaluate("Random Forest Regressor", y_test, y_pred_rf)

### 3.6 Model Comparison

In [ ]:
comparison = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'R² Score'],
    'Linear Regression': [f'{mae_lr:,.0f}', f'{rmse_lr:,.0f}', f'{r2_lr:.4f}'],
    'Random Forest'    : [f'{mae_rf:,.0f}', f'{rmse_rf:,.0f}', f'{r2_rf:.4f}']
})
print("\n📊 Model Comparison")
print(comparison.to_string(index=False))
winner = "Linear Regression" if r2_lr > r2_rf else "Random Forest"
print(f"\n🏆 Better model: {winner}")


📊 Model Comparison
  Metric Linear Regression Random Forest
     MAE           979,680     1,011,501
    RMSE         1,331,071     1,388,576
R² Score            0.6495        0.6185

🏆 Better model: Linear Regression


---
## 📊 Task 4 — Visualizations

### Chart 1 — Distribution of House Prices

In [ ]:
import os
os.makedirs('charts', exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['price'], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(df['price'].median(), color='tomato', linestyle='--', lw=2,
                label=f'Median: {df["price"].median()/1e6:.2f}M')
axes[0].axvline(df['price'].mean(), color='gold', linestyle='--', lw=2,
                label=f'Mean: {df["price"].mean()/1e6:.2f}M')
axes[0].set_title('House Price Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Price (₹)')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].hist(np.log(df['price']), bins=40, color='mediumseagreen', edgecolor='white', alpha=0.85)
axes[1].set_title('Log-Scale Price Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('log(Price)')
axes[1].set_ylabel('Count')

plt.suptitle('Chart 1 — House Price Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('charts/chart1_price_distribution.png', bbox_inches='tight')
plt.show()
print("✅ Saved: charts/chart1_price_distribution.png")

### Chart 2 — Correlation Heatmap

In [ ]:
plt.figure(figsize=(13, 9))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(230, 20, as_cmap=True)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap=cmap,
            center=0, linewidths=0.5, linecolor='white',
            annot_kws={"size": 9}, square=True)
plt.title('Chart 2 — Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('charts/chart2_correlation_heatmap.png', bbox_inches='tight')
plt.show()
print("✅ Saved: charts/chart2_correlation_heatmap.png")

print("\nTop correlations with price:")
print(corr['price'].drop('price').sort_values(ascending=False).to_string())

### Chart 3 — Actual vs Predicted Prices & Feature Importance

A two-panel chart: left shows how closely model predictions match actual prices; right shows which features the Random Forest found most important.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel A — Actual vs Predicted
ax = axes[0]
ax.scatter(y_test, y_pred_rf, alpha=0.55, color='royalblue', edgecolors='white',
           linewidth=0.4, s=60, label='Random Forest')
ax.scatter(y_test, y_pred_lr, alpha=0.45, color='tomato', edgecolors='white',
           linewidth=0.4, s=40, label='Linear Regression', marker='^')
lims = [min(y_test.min(), y_pred_rf.min()), max(y_test.max(), y_pred_rf.max())]
ax.plot(lims, lims, 'k--', lw=1.5, label='Perfect Prediction')
ax.set_xlabel('Actual Price (₹)', fontsize=11)
ax.set_ylabel('Predicted Price (₹)', fontsize=11)
ax.set_title('Actual vs Predicted House Prices', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.text(0.05, 0.93, f'LR  R² = {r2_lr:.3f}', transform=ax.transAxes,
        fontsize=10, color='tomato',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7))
ax.text(0.05, 0.85, f'RF  R² = {r2_rf:.3f}', transform=ax.transAxes,
        fontsize=10, color='royalblue',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7))

# Panel B — Feature Importance
ax2 = axes[1]
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)
colors = ['#4c72b0' if v > importances.median() else '#a9c4e0' for v in importances]
importances.plot(kind='barh', ax=ax2, color=colors, edgecolor='white')
ax2.set_title('Random Forest — Feature Importance', fontsize=13, fontweight='bold')
ax2.set_xlabel('Importance Score', fontsize=11)
ax2.axvline(importances.median(), color='tomato', linestyle='--', lw=1.2, label='Median')
ax2.legend(fontsize=9)

plt.suptitle('Chart 3 — Model Performance & Feature Importance',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('charts/chart3_model_performance.png', bbox_inches='tight')
plt.show()
print("✅ Saved: charts/chart3_model_performance.png")

### Chart 4 (Bonus) — House Price vs Area by Air Conditioning

In [ ]:
plt.figure(figsize=(11, 6))
for val, label, color in [(1,'With AC','royalblue'), (0,'Without AC','lightcoral')]:
    mask = df['airconditioning'] == val
    plt.scatter(df.loc[mask,'area'], df.loc[mask,'price'],
                alpha=0.55, c=color, label=label, s=50, edgecolors='white', lw=0.3)
plt.title('Bonus Chart 4 — House Price vs Area (by Air Conditioning)',
          fontsize=13, fontweight='bold')
plt.xlabel('Area (sq ft)', fontsize=12)
plt.ylabel('Price (₹)', fontsize=12)
plt.legend(fontsize=10)
plt.tight_layout()
plt.savefig('charts/chart4_price_vs_area.png', bbox_inches='tight')
plt.show()
print("✅ Saved: charts/chart4_price_vs_area.png")

---
## 📝 Task 5 — Insights & Summary

### Which features influence house price the most?

Based on the **correlation heatmap** from the actual dataset, the top price drivers are:

| Feature | Correlation with Price | Interpretation |
|---|---|---|
| Area (sq ft) | **0.54** | Larger homes → higher prices — the single strongest predictor |
| Bathrooms | **0.52** | More bathrooms → reflects quality and overall property size |
| Air Conditioning | **0.45** | AC is treated as a premium feature in this market |
| Stories | **0.42** | Multi-storey homes command higher prices |
| Parking | **0.38** | Dedicated parking adds meaningful value |
| Bedrooms | **0.37** | Moderate effect — buyers value quality over bedroom count |
| Hot Water Heating | **0.09** | Weakest link — treated as a basic expectation, not a premium |

---

### How accurate was the model (in plain terms)?

The **Linear Regression** model was the better performer on this dataset with an **R² of 0.6495**, meaning it explains approximately **65% of the variation** in house prices. With a MAE of ₹9,79,680 and RMSE of ₹13,31,071 on a price range of ₹17.5L–₹1.33Cr, predictions are useful for ballpark valuations but not precise enough for final pricing decisions.

Notably, **Linear Regression outperformed Random Forest** (R² 0.6495 vs 0.6185) — which is unusual. This tells us the real dataset has a relatively linear structure, so the simpler model was sufficient.

---

### What surprised you in the data?

Two things were surprising:
1. **Random Forest did not beat Linear Regression** — on most housing datasets, the ensemble model wins. The clean, well-structured Kaggle dataset made the linear relationship strong enough for LR to win.
2. **Hot water heating had almost no correlation with price (0.09)** — despite being a practical amenity. Buyers appear to treat it as a basic expectation, while features like AC and preferred area are treated as true price premiums.

---

### One recommendation for a real estate business

> **Focus investment on properties with large area, multiple bathrooms, and air conditioning — in that priority order.** These three features account for the highest correlation with price. Since the market shows a linear pricing relationship, a simple pricing formula based on these three variables can produce reliable estimates without needing complex models. Additionally, given that hot water heating adds almost no price value, developers should redirect that budget toward AC installation or larger floor plans.

---
*Project completed ✅ — All 5 tasks done. Charts saved in `/charts` folder.*